# 20 - Pass 10 Q2 + Q3: is the MLM gain stable, and how many labels does it save? (one commit, GPU)

**Q2, pretraining seeds:** Pass 8a pretrained MLM once (seed 0). This notebook pretrains it **twice more**
(seeds 1 and 2, every other setting the same), one per GPU at the same time, then fine-tunes A20 from each
at 1% and 10% labels, 5 seeds (tags `pt1`, `pt2`; 20 runs). Three encoders show whether +7.4 is stable.

**Q3, the label curve:** A20 (seed-0 encoder) at 5% and 30% labels, and with `RUN_FULL100` at 100%
(15 runs). Plain BERT (A1) exists at every budget, so the notebook can say how many labels MLM saves.

**Q5 (optional, `RUN_LONG = False` by default):** one 20,000-step pretraining, then A20 at 1% and 10%
(tag `pt20k`). A dose check only: the paper keeps the 10,000-step encoder whatever it shows. Adds about 1.5 hours.

**Nothing is picked on test.** About 2-2.5 hours with the 100% runs (pretraining about 40 minutes).

**Before you run, in the right-hand panel:**
- **Input 1:** the same `gainsplit-processed` version as notebooks 10-18.
- **Input 2:** the output of **notebook 15** (Add Input -> Your Work -> Notebooks -> `15-pass8-mlm`, the
  version that made `results/pass8_mlm/`). It holds the seed-0 MLM encoder (`ckpt/mlm`).
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**; you can close the browser.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
The repo's tests train a tiny random BERT for a few steps on made-up data. For Pass 10 they also train
the six new arms from a tiny pretrained encoder, check the self-training teacher starts from that encoder,
check the encoder finder refuses an encoder whose weights differ from the receipt, and run the analysis
on stand-in runs. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Find the MLM encoder from notebook 15 and check it is the same one
Every Pass 10 comparison is against Pass 8a's A20/A21 runs, so the new runs must start from **exactly the
encoder those runs used**. This cell looks for it in the attached notebook-15 output and compares its weight
hashes with the receipt committed in `results/pass8_mlm/pretrain_info.json`. If it is missing or different,
the cell stops: attach the right input (see the top of this notebook) rather than pretraining again.

In [ ]:
from gainsplit.ssl_pretrain import find_encoder
receipt = json.loads(Path(f"{REPO_DIR}/results/pass8_mlm/pretrain_info.json").read_text())
ENC = str(find_encoder("/kaggle/input", "mlm", seed=0, steps=10000, expected_sha256=receipt["weights_sha256"]))
print("MLM encoder (seed 0, 10,000 steps):", ENC, "- weights match the Pass 8a receipt")

## 4. Check the data on the CPU (pre-flight)
`RUN_FULL100 = True` adds A20 at 100% (5 runs, about 50 minutes of run time). `RUN_LONG = True` adds Q5.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS, PASS10_CKPT, PASS10_PRETRAIN_SEEDS, PASS10_LONG_STEPS

PLAN = "pass10_seeds"
RUN_FULL100 = True
RUN_LONG = False
OUT_DIR = f"/kaggle/working/results/{PLAN}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
configs = ((PLANS["pass10_curve_100"](ENC) if RUN_FULL100 else []) + PLANS["pass10_curve"](ENC)
           + PLANS["pass10_seeds"]() + (PLANS["pass10_long"]() if RUN_LONG else []))
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))

## 5. Pretrain MLM with seeds 1 and 2, one per GPU at the same time (about 40 minutes)
Masked-word prediction on the 22,800 training headlines, **no labels**, exactly Pass 8a's settings except the
seed (it changes which words are hidden and the batch order). Each pretraining also runs the frozen linear
probe before and after. Receipts go into the zip (`pretrain_info_pt1.json`, `pretrain_info_pt2.json`).
If the session restarts, finished pretrainings are skipped.

In [ ]:
def pretrain_jobs(jobs):
    """jobs: list of (name, gpu, extra config). Runs them at the same time, one per GPU; waits for all."""
    procs = []
    for name, gpu, extra in jobs:
        env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src", CUDA_VISIBLE_DEVICES=str(gpu))
        log = open(f"{OUT_DIR}/pretrain_{name}.log", "w")
        p = subprocess.Popen([sys.executable, "-m", "gainsplit.ssl_pretrain", "--objective", "mlm",
                              "--data", str(DATA_DIR), "--out", PASS10_CKPT[name], "--config", json.dumps(extra)],
                             stdout=log, stderr=subprocess.STDOUT, env=env)
        procs.append((name, p, log))
    for name, p, log in procs:
        p.wait(); log.close()
        print(open(f"{OUT_DIR}/pretrain_{name}.log").read()[-1500:])
        assert p.returncode == 0, f"pretraining {name} failed: see its log above"
        shutil.copy(f"{PASS10_CKPT[name]}/pretrain_info.json", f"{OUT_DIR}/pretrain_info_{name}.json")
        info = json.loads(Path(f"{OUT_DIR}/pretrain_info_{name}.json").read_text())
        print(f"{name}: {info['minutes']} min; dev loss {info['dev_before']['loss']} -> {info['dev_after']['loss']}; "
              f"probe 1% {info['probe_before']['p1']['dev_macro_f1_mean']:.2f} -> {info['probe_after']['p1']['dev_macro_f1_mean']:.2f}")

t0 = time.time()
pretrain_jobs([(f"pt{k}", i % max(1, torch.cuda.device_count()), {"seed": k}) for i, k in enumerate(PASS10_PRETRAIN_SEEDS)])
if RUN_LONG:
    pretrain_jobs([("pt20k", 0, {"seed": 0, "steps": PASS10_LONG_STEPS})])
print(f"pretraining: {(time.time() - t0) / 60:.1f} min")

## 6. The list of runs
A20 at 100% first (about 10 min each), then 30%, 5%, then the pretraining-seed runs at 10% and 1%.

In [ ]:
for c in configs:
    print(f"{c.run_id:20s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} init={c.init_dir}")
print(len(configs), "runs")

## Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`.
If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## Checks and analysis (CPU, about 3 minutes)
1. **Pretraining receipts** for the three (or four) encoders: dev loss and linear probe before and after.
2. **Seed spread:** A20's mean test macro-F1 per encoder, and the range across encoders.
3. **Curve:** A1 and A20 per budget, and the A1 budget that A20's score matches (log-linear interpolation;
   `label_factor` = how many times fewer labels MLM needs for the same score). A reading aid, no test.
4. **Paired bootstrap**, Holm within each table: `seeds_f1` (each new encoder vs A1, and the mean of the three),
   `curve_f1` (A20 vs A1 at 5/30/100%, and cross-budget rows such as MLM with 1% vs BERT with 10%),
   `long_f1` (only with `RUN_LONG`).

In [ ]:
from gainsplit import analysis_pass10 as a10
import pandas as pd
pd.set_option("display.width", 200)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass2a", "pass8_mlm")] + [OUT_DIR]
                if Path(d).exists()]
print("results folders:", RESULTS_DIRS)
infos = [f"{REPO_DIR}/results/pass8_mlm/pretrain_info.json"] + [f"{OUT_DIR}/pretrain_info_{n}.json"
                                                                for n in ("pt1", "pt2", "pt20k")]
tables = a10.run_all(RESULTS_DIRS, "seeds_curve", f"{OUT_DIR}/analysis", pretrain_infos=[p for p in infos if Path(p).exists()])
for name, t in tables.items():
    if name != "verdicts":
        print(f"\n=== {name}")
        display(t)

## Verdict against the predictions written before running
Each prediction counts as **supported** only if the result goes the predicted way **and** passes Holm
(p_holm < 0.05); "no difference" predictions are supported when p_holm >= 0.05 (which is not proof of
equality: read the interval). Predictions and reasons: vault, Experiment Plan, "Pass 10 plan".
Paste this cell's output when you report.

In [ ]:
v = tables["verdicts"]
for _, r in v.iterrows():
    if pd.isna(r.get("mean_diff")):
        print(f"{r.hypothesis} {r.contrast} at {r.budget}%: {r.verdict}")
        continue
    print(f"{r.hypothesis} {r.contrast:44s} {r.budget:3d}%  {r.mean_diff:+.2f} points [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  -> {r.verdict}   ({r.claim})")

## Download
Zips the results folder (runs, logs, receipts, analysis tables). A commit keeps it in the Output tab:
download `results_<plan>.zip` and send it with the notebook log.

In [ ]:
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")